# Best-of-N Speed Benchmark Across Models

Benchmark Best-of-N generation speed on the same set of MATH
problems across multiple large language models.

Each model is loaded with vLLM, warmed up, timed for
`num_trials` runs, and then unloaded before the next model is
tested. This keeps the comparison focused on model-level
throughput under the same benchmark settings.

Use this notebook to compare speed across model families or model
sizes. Use `benchmark_speed_bon_quant_v1.ipynb` for the separate
quantization sweep.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

import statistics

from utils.configs import GenConfig

from utils.load_data import load_data_hf
from unittests.notebook_utils import gpu_mem_used_gb, benchmark_bon_speed_llm_model

In [2]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = f"{base_dir}/prm800k/math_splits"

# Models to benchmark — same prompts, same config; only this varies
llm_dirs = [
    f"{base_dir}/Llama3.2-1B-Instruct",
    f"{base_dir}/Llama3.2-3B-Instruct",
    # f"{base_dir}/Llama3.2-7B-Instruct",
    f"{base_dir}/Qwen2.5-Math-1.5B-Instruct",
    f"{base_dir}/Qwen2.5-Math-7B-Instruct",
]

In [3]:
# Best-of-N search params
config = GenConfig()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs (all in one place so the matrix is easy to vary)
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 10                 # cap on questions per benchmark
num_trials = 1                     # timed runs per model
warmup = 1                         # untimed warmup runs per model
llm_gpu_memory_utilization = 0.7

In [4]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 10


## Run benchmark

One model at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [ ]:
results = []
for llm_dir in llm_dirs:
    name, times = benchmark_bon_speed_llm_model(
        llm_dir, config, batch_of_questions, num_trials,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        warmup=warmup,
    )
    results.append((name, times))


=== Llama3.2-1B-Instruct ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.32s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.32s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used: 22.84 GB
  trial 0:   38.32s total, 3.8321s/question


[rank0]:[W616 11:25:45.627123206 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== Llama3.2-3B-Instruct ===


## Summary

In [ ]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'model':<24}{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<24}{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )